# 01 - CICIDS2017: Clean Balanced Subset Generation

### Why Create a Subset?
1. **Speed & DL Iteration**: Full CICIDS has ~2.2M rows. Training takes minutes to hours. A ~170k subset trains in seconds and enables rapid Deep Learning experimentation.
2. **Fixes Overwhelming BENIGN Dominance**: In the full dataset, BENIGN is 85%. By capping BENIGN to ~100k and keeping all rare attacks, the ratio becomes ~59% Benign : 41% Attacks, making the task challenging and realistic.
3. **100% Preservation of Rare Attacks**: All Web Attacks, Bots, PortScans, and Brute Force flows are preserved.
4. **Outlier & Redundancy Cleaned**: Drops duplicates, zero-variance features, and collinear features (|r| > 0.95).

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import time
from sklearn.feature_selection import VarianceThreshold

import warnings
warnings.filterwarnings('ignore')

## 1. Load All 8 CSVs, Clean Identifiers, NaNs, and Duplicates

In [ ]:
files = glob.glob(r'../Datasets/CICIDS/*.csv')
print(f"Found {len(files)} CSV files.")

df_list = []
for f in files:
    sub_df = pd.read_csv(f)
    df_list.append(sub_df)

df = pd.concat(df_list, ignore_index=True)
df.columns = df.columns.str.strip()
df['Label'] = df['Label'].astype(str).str.strip().str.encode('ascii', 'ignore').str.decode('utf-8')
df['Label'] = df['Label'].replace('', 'UNKNOWN')

drop_cols = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Timestamp', 'Destination Port']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df = df.replace([np.inf, -np.inf], np.nan).dropna()

original_len = len(df)
df = df.drop_duplicates()
print(f"Dropped {original_len - len(df)} duplicate rows.")
print("Deduplicated Dataset Shape:", df.shape)

## 2. Group Attack Classes into Meta-Categories

In [ ]:
label_map = {
    'Web Attack  Brute Force': 'Web Attack',
    'Web Attack  XSS': 'Web Attack',
    'Web Attack  Sql Injection': 'Web Attack',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
}
df['Label'] = df['Label'].replace(label_map)

min_samples = 50
class_counts = df['Label'].value_counts()
valid_labels = class_counts[class_counts >= min_samples].index
removed_labels = class_counts[class_counts < min_samples]
if len(removed_labels) > 0:
    print(f"Removing classes with < {min_samples} samples: {list(removed_labels.index)}")
df = df[df['Label'].isin(valid_labels)]

print("Full Class Distribution:")
print(df['Label'].value_counts())

## 3. Feature Selection: Remove Zero-Variance & Collinear Features (|r| > 0.95)

In [ ]:
X = df.drop(columns=['Label'])
y = df['Label']

vt = VarianceThreshold(threshold=0.01)
X_filtered = pd.DataFrame(vt.fit_transform(X), columns=X.columns[vt.get_support()], index=X.index)
removed_variance = set(X.columns) - set(X_filtered.columns)
print(f"Removed {len(removed_variance)} near-zero variance features.")
X = X_filtered

corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
print(f"Removed {len(high_corr_cols)} highly correlated features (|r| > 0.95).")
X = X.drop(columns=high_corr_cols)
print(f"Remaining clean features: {X.shape[1]}")

df_clean = pd.concat([X, y], axis=1)

## 4. Balanced Stratified Sampling (~170,000 Rows)

- Keep 100% of all rare attacks (Bot, Web Attack, PortScan, Brute Force)
- Cap DoS to 30,000 and DDoS to 25,000
- Sample BENIGN to 100,000

In [ ]:
sample_caps = {
    'BENIGN': 100000,
    'DoS': 30000,
    'DDoS': 25000,
    'Brute Force': 9150,
    'Web Attack': 2143,
    'PortScan': 1956,
    'Bot': 1437
}

sampled_dfs = []
for label, max_count in sample_caps.items():
    sub = df_clean[df_clean['Label'] == label]
    if len(sub) > max_count:
        sub = sub.sample(n=max_count, random_state=42)
    sampled_dfs.append(sub)

df_subset = pd.concat(sampled_dfs, ignore_index=True)
df_subset = df_subset.sample(frac=1, random_state=42).reset_index(drop=True)

print("Subset Shape:", df_subset.shape)
print("\nSubset Class Distribution:")
print(df_subset['Label'].value_counts())
print("\nPercentage Distribution:")
print((df_subset['Label'].value_counts(normalize=True) * 100).round(2).astype(str) + '%')

## 5. Save Clean Subset (Parquet & CSV)

In [ ]:
output_dir = r'../Datasets'
parquet_path = os.path.join(output_dir, 'CICIDS_clean_subset.parquet')
csv_path = os.path.join(output_dir, 'CICIDS_clean_subset.csv')

print("Saving parquet format (fast & compressed)...")
df_subset.to_parquet(parquet_path, index=False)
print(f"Saved: {parquet_path} ({os.path.getsize(parquet_path) / (1024*1024):.2f} MB)")

print("\nSaving CSV format...")
df_subset.to_csv(csv_path, index=False)
print(f"Saved: {csv_path} ({os.path.getsize(csv_path) / (1024*1024):.2f} MB)")

## 6. Verify Loading Speed

In [ ]:
start = time.time()
test_load = pd.read_parquet(parquet_path)
elapsed = time.time() - start
print(f"Loaded {test_load.shape[0]:,} rows and {test_load.shape[1]} columns from parquet in {elapsed:.2f} seconds!")